# **Importing the needed libraries**

In [ ]:
from pyspark.sql import SparkSession, functions as F, types as T
from pyspark.ml import Pipeline
from pyspark.ml.feature import Imputer, StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.sql.functions import col, udf
from pyspark.sql.types import DoubleType

# **Setup of spark**

In [ ]:
# Start a Spark session
spark = SparkSession.builder \
    .appName("Satisfaction_Classification") \
    .getOrCreate()
# Load the merged dataset into a Spark DataFrame
merged = spark.read.csv("/content/merged.csv", header=True, inferSchema=True)

# **data exploration**

In [ ]:
#schema of each column
merged.printSchema()
#printing the first ten rows of the data
merged.show(10, truncate=False)

In [ ]:
#checking label class balance
valid_classes = ["extremely satisfied", "satisfied", "not satisfied"]
merged = merged.filter(merged.satisfaction_category.isin(valid_classes))
merged.groupBy("satisfaction_category").count().show()

In [ ]:
# checking for Missing values
null = merged.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in merged.columns])
null.show()

# **Data cleaning and preprocessing**

In [ ]:
#handeling null values
#dropping review_comment_title and review_comment_message because they have a huge number of null values so they won't add anything to classification
df = merged.drop("review_comment_title", "review_comment_message")

In [ ]:
#dropping the columns that act as noise and will not add anything to the classification model
#all the id columns are dropped as they are unique per row so they have no effect on the classification
#we already created features like delay, so raw dates are dropped to avoid redundancy
#bin and grouped columns are only added to the data for visualizations so they are dropped
columns_drop = [
    "customer_id", "review_id", "order_item_id", "product_id", "seller_id",
    "order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
    "order_delivered_customer_date", "order_estimated_delivery_date", "shipping_limit_date",
    "review_creation_date", "review_answer_timestamp",
    "price_bin", "freight_bin", "num_items_group","product_category_name_english","review_score"
]

df = df.drop(*columns_drop)

In [ ]:
#defining the numerical columns to cast them to double
numeric_to_double = [
    "price", "freight_value", "product_name_lenght", "product_description_lenght",
    "product_photos_qty", "product_weight_g", "product_length_cm", "product_height_cm",
    "product_width_cm", "num_items", "total_price", "total_freight", "delay_num_days"
]
#casting to double
for column_name in numeric_to_double:
    df = df.withColumn(column_name, F.col(column_name).cast("double"))

In [ ]:
#the target column was unbalnaced so i attempted to balnace it based on setting weights for each label but it has resduced the results of the model
#balnacing the target column
# Count rows for each satisfaction category
#class_c = df.groupBy("satisfaction_category").count().collect()
#total_c = df.count()
# create dictionary (category: weight)
#class_weights = {row["satisfaction_category"]: total_c / row["count"] for row in class_c}
#print("Class Weights:", class_weights)
# define UDF to assign weight based on the satisfaction_category
#def get_weight(cat):
#    return float(class_weights[cat])

#weight_udf = udf(get_weight, DoubleType())
# add the weight column to df
#df = df.withColumn("classWeightCol", weight_udf(col("satisfaction_category")))

# **Preparing the components of ML pipline**

In [ ]:
#encoding categorical columns
#columns name before
index_cols = ["order_status", "delay_category"]
#columns name after
indexed_cols = ["order_status_index", "delay_category_index"]
#encoding stage to add into the pipeline
ind = [StringIndexer(inputCol=column, outputCol=column+"_index", handleInvalid="keep") for column in index_cols]

In [ ]:
#encoding the label stage to be used in pipline
label_indexer = StringIndexer(inputCol="satisfaction_category",outputCol="label",handleInvalid="skip")

In [ ]:
#defining the features columns without the label
feature_cols = numeric_to_double + indexed_cols
#defining assembler that Combines all the columns into one vector, because spark ml models expects one column
assembler = VectorAssembler(inputCols=feature_cols,outputCol="features",handleInvalid="skip")

In [ ]:
#defining imputer that is used to handle null values in numeric columns, the strategy here is filling with mean
imputer = Imputer(inputCols=numeric_to_double,outputCols=numeric_to_double).setStrategy("mean")

In [ ]:
#defining evaluation metrices
eval_acc = MulticlassClassificationEvaluator(metricName="accuracy",  labelCol="label", predictionCol="predicted_label")
eval_f1 = MulticlassClassificationEvaluator(metricName="f1",        labelCol="label", predictionCol="predicted_label")
eval_prec = MulticlassClassificationEvaluator(metricName="weightedPrecision", labelCol="label", predictionCol="predicted_label")
eval_rec = MulticlassClassificationEvaluator(metricName="weightedRecall",    labelCol="label", predictionCol="predicted_label")

In [ ]:
#defining hyperparameters for grid search
grid_numTrees = [50, 100, 150]
grid_maxDepth = [5, 8, 10]

In [ ]:
#splitting for testiung and training
#training is 60% and training is 40%
train, test = df.randomSplit([0.6, 0.4], seed=42)

# **Classification using Randomforest Pipeline**




In [ ]:
#list to save results of the diffirant hyperparamter combinations
results = []
# loop through different combinations of hyperparameters
for numTrees in grid_numTrees:
    for maxDepth in grid_maxDepth:
            # Define RF classifier that has the current iteration hyperparameter
            rf = RandomForestClassifier(
                labelCol="label",
                featuresCol="features",
                numTrees=numTrees,
                maxDepth=maxDepth,
                maxBins=128,
                seed=42
                #balnacing the data based on weights for every label but it was removed cause it reduced the accuracy of the model
                #weightCol="classWeightCol"
            )

            # building pipeline
            pipeline = Pipeline(stages=ind + [label_indexer, imputer, assembler, rf])
            # Trainning the model on trianing data
            model = pipeline.fit(train)
            # Predicting on testing dat
            pred = model.transform(test)
            pred = pred.withColumnRenamed("prediction", "predicted_label")
            # Evaluation
            acc  = eval_acc.evaluate(pred)
            f1   = eval_f1.evaluate(pred)
            precision = eval_prec.evaluate(pred)
            recall  = eval_rec.evaluate(pred)
            #adding the results to the list to find best model
            results.append({
                "numTrees": numTrees,
                "maxDepth": maxDepth,
                "Accuracy": acc,
                "F1": f1,
                "Precision": precision,
                "Recall": recall
            })

# Convert results to Pandas for easy viewing
import pandas as pd
df_results = pd.DataFrame(results)
print(df_results)

# Save predictions from the best model
pred.select(
    "order_id", "label", "predicted_label", "probability"
).toPandas().to_csv("predictions.csv", index=False)

# **saving the predictions with the original merged data**

In [ ]:
import pandas as pd
#  original dataset with all the columns
original = pd.read_csv("merged.csv")
#  predictions
preds = pd.read_csv("predictions.csv")
# merge based on order_id
final = pd.merge(original, preds, on="order_id", how="left")
# save full file
final.to_csv("satisfaction_predictions.csv", index=False)